In [ ]:
import sys
import numpy as np
import pandas as pd
from collections import defaultdict
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly

CHANNELS: dict[str, tuple[str]] = {
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.DXY, Sources.UST_10Y),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
  "Domestic Baseline": (Sources.cpi_inflation,)
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

full_models, fit_dict, partialr2_dict = {}, {}, {}
wald_dict = defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full

  fit_dict[d_var] = {
    ("Full R2"): round(m_full.rsquared, 4),
    ("Adj. R2"): round(m_full.rsquared_adj, 4),
  }

  partialr2_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      rhs_drop = [x for x in ALL_PREDICTORS if x != var] # exclude the current predictor 
      m_drop = fit_hac_model(d_var, rhs_drop, data)
      partialr2_dict[d_var][(ch_name, var)] = round(m_full.rsquared - m_drop.rsquared, 4)

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)

    wald_dict[ch_name][(d_var, "F-Stat")] = round(float(w_res.statistic), 4)
    wald_dict[ch_name][(d_var, "p-value")] = round(float(w_res.pvalue), 4)

df_fit_matrix = pd.DataFrame(fit_dict)
df_wald_matrix = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict)

### 1. ADF (Stationary), VIF (Multicollinearity)

In [ ]:
print("=" * 45)
print("SECTION 1: DIAGNOSTIC ASSUMPTIONS (ADF, VIF)")
print("=" * 45)

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "stat"): round(float(stat), 4),
    ("ADF", "p-val"): round(float(p), 4)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = round(float(vif_score), 4)

diagnostic_dict = pd.DataFrame(diagnostic_dict).T
diagnostic_dict.columns.names = ["Test", ""]
print(diagnostic_dict)

SECTION 1: DIAGNOSTIC ASSUMPTIONS (ADF, VIF)
Test                     ADF             VIF
                        stat   p-val   score
FFR_OPR_diff         -4.7400  0.0001  1.1354
VIX                  -5.6326  0.0000  1.1336
DXY                 -11.4835  0.0000  1.2492
UST_10Y             -10.8911  0.0000  1.3767
CPO                  -9.1417  0.0000  1.0369
Oil                  -8.5870  0.0000  1.4507
cpi_inflation        -9.3565  0.0000  1.0674
KLCI                -13.1469  0.0000     NaN
financials          -10.4197  0.0000     NaN
energy              -11.0151  0.0000     NaN
industrial_products -10.0120  0.0000     NaN
plantation          -13.8697  0.0000     NaN
REITs               -14.8697  0.0000     NaN
technology           -4.6504  0.0001     NaN


### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [3]:
print("\n" + "=" * 95)
print("SECTION 4: MODEL FIT & CHANNEL VARIANCE DECOMPOSITION (PARTIAL R²)")
print("=" * 95)
print(df_fit_matrix.to_string())
print("\nPartial r2 breakdown by channel:")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())


SECTION 4: MODEL FIT & CHANNEL VARIANCE DECOMPOSITION (PARTIAL R²)
           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Full R2  0.3305      0.2741  0.2065               0.1333      0.1235  0.1888      0.2099
Adj. R2  0.2948      0.2353  0.1642               0.0870      0.0602  0.1455      0.1677

Partial r2 breakdown by channel:
                           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Rate Differential        0.0011      0.0000  0.0027               0.0032      0.0347  0.0247      0.0010
Global Risk Appetite     0.0767      0.0554  0.0000               0.0161      0.0002  0.0359      0.1646
Global Credit Condition  0.1045      0.1097  0.0193               0.0322      0.0264  0.0593      0.0007
Commodity                0.0163      0.0152  0.0852               0.0130      0.0509  0.0516      0.0081
Domestic Baseline        0.0132      0.0010  0.0230               0.0211      0.0002  0.0008      0.0001


### 5. INFERENCE & FEATURE IMPORTANCE

In [4]:
print("\n" + "=" * 95)
print("SECTION 5A: CHANNEL WALD JOINT SIGNIFICANCE TESTS (F-STAT & P-VALUES)")
print("=" * 95)
print(df_wald_matrix.to_string())



SECTION 5A: CHANNEL WALD JOINT SIGNIFICANCE TESTS (F-STAT & P-VALUES)
                            KLCI         financials          energy         industrial_products         plantation           REITs         technology        
                          F-Stat p-value     F-Stat p-value  F-Stat p-value              F-Stat p-value     F-Stat p-value  F-Stat p-value     F-Stat p-value
Rate Differential         0.1505  0.6987     0.0011  0.9732  0.4908  0.4848              0.2759  0.6003     4.2081  0.0429  2.3459  0.1280     0.3147  0.5757
Global Risk Appetite     16.1237  0.0001    10.8904  0.0012  0.0013  0.9708              2.7484  0.0997     0.0293  0.8644  6.0875  0.0149    15.7085  0.0001
Global Credit Condition   9.2165  0.0002     6.0346  0.0031  1.3803  0.2551              2.4042  0.0943     1.6127  0.2046  4.3344  0.0150     0.0639  0.9381
Commodity                 1.6704  0.1921     1.7176  0.1835  1.6867  0.1891              0.8736  0.4199     6.2490  0.0028  3.4967  0.0332 